<a href="https://colab.research.google.com/github/Matheusbcy/-Data-Science-IA-/blob/main/Deep%20Learning/Projetos/Regress%C3%A3o/Previs%C3%A3o_IMDB_filmes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
df = pd.read_csv('desafio_indicium_imdb.csv')
df.drop(labels = ["Unnamed: 0"], axis = 1, inplace = True)

In [ ]:
df.columns

Index(['Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre',
       'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2',
       'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
Series_Title,0
Released_Year,0
Certificate,101
Runtime,0
Genre,0
IMDB_Rating,0
Overview,0
Meta_score,157
Director,0
Star1,0


In [ ]:
df.dropna(inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 0 to 996
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Series_Title   713 non-null    object 
 1   Released_Year  713 non-null    object 
 2   Certificate    713 non-null    object 
 3   Runtime        713 non-null    object 
 4   Genre          713 non-null    object 
 5   IMDB_Rating    713 non-null    float64
 6   Overview       713 non-null    object 
 7   Meta_score     713 non-null    float64
 8   Director       713 non-null    object 
 9   Star1          713 non-null    object 
 10  Star2          713 non-null    object 
 11  Star3          713 non-null    object 
 12  Star4          713 non-null    object 
 13  No_of_Votes    713 non-null    int64  
 14  Gross          713 non-null    object 
dtypes: float64(2), int64(1), object(12)
memory usage: 89.1+ KB


In [ ]:
df = df[df['Released_Year'] != 'PG']

In [ ]:
df['Released_Year'] = pd.to_numeric(df['Released_Year'], errors='coerce')

In [ ]:
df.isna().sum()

,0
Series_Title,0
Released_Year,0
Certificate,0
Runtime,0
Genre,0
IMDB_Rating,0
Overview,0
Meta_score,0
Director,0
Star1,0


In [ ]:
def modifies_runtime(runtime_str):
    runtime_str = runtime_str.lower()

    runtime_str = re.sub(r'[^0-9]', '', runtime_str)

    return runtime_str

In [ ]:
# MOdiciando valores da coluna Runtime
df["Runtime"] = df["Runtime"].apply(modifies_runtime).astype(float)

# Treinamento de Modelo

In [ ]:
df.drop(labels = ["Series_Title", "Overview"], axis = 1, inplace = True)

In [ ]:
df.head()

,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,1972,A,175.0,"Crime, Drama",9.2,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
1,2008,UA,152.0,"Action, Crime, Drama",9.0,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
2,1974,A,202.0,"Crime, Drama",9.0,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
3,1957,U,96.0,"Crime, Drama",9.0,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"
4,2003,U,201.0,"Action, Adventure, Drama",8.9,94.0,Peter Jackson,Elijah Wood,Viggo Mortensen,Ian McKellen,Orlando Bloom,1642758,"377,845,905"


In [ ]:
df['Gross'] = df['Gross'].str.replace(',', '').astype(float)

In [ ]:
X = df.drop(columns = ["IMDB_Rating"])

categorical_cols = ["Certificate", "Genre", "Director", "Star1", "Star2", "Star3", "Star4"]
numeric_cols = ["Released_Year", "Runtime", "Meta_score", "No_of_Votes", "Gross"]

preprocessor = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

X_transformed = preprocessor.fit_transform(X)

In [ ]:
feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = list(feature_names) + numeric_cols

In [ ]:
X_transformed_df = pd.DataFrame(X_transformed.toarray() if hasattr(X_transformed, "toarray") else X_transformed, columns = all_feature_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed, df["IMDB_Rating"], test_size = 0.2, random_state = 1)

In [ ]:
X_train.shape, X_test.shape

((569, 2953), (143, 2953))

# Treinamento da Rede Neural

In [ ]:
!pip install scikeras
!pip install scikit-learn==1.5.2

In [ ]:
import scikeras
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from tensorflow.keras import backend as K
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from tensorflow.keras.layers import Dropout
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

In [ ]:
def criar_rede_dropout(meta, hidden_layers=2, neurons=64, activation='relu',
                      kernel_initializer='he_uniform', dropout_rate=0.2,
                      optimizer='adam', loss='mse'):
    K.clear_session()

    n_features = meta["X_shape_"][1]

    model = Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))

    for i in range(hidden_layers):
        model.add(Dense(units=neurons, activation=activation,
                       kernel_initializer=kernel_initializer))
        model.add(Dropout(rate=dropout_rate))

    model.add(Dense(units=1, activation="linear"))

    model.compile(optimizer=optimizer,
                  loss=loss,
                  metrics=["mae"])
    return model

model = KerasRegressor(
    model=criar_rede_dropout,
    hidden_layers=2,
    verbose=0
)

pipeline = Pipeline([
    ("pca", PCA()),
    ("model", model)
])

param_grid = {
    "pca__n_components": [50, 100, 150],
    "model__model__neurons": [32, 64],
    "model__model__activation": ["relu", "tanh"],
    "model__model__dropout_rate": [0.2, 0.3],
    "model__optimizer": ["adam", "rmsprop"],
    "model__batch_size": [32],
    "model__epochs": [100]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    verbose=2
)

grid_search.fit(X_train, y_train_scaled.ravel())

In [ ]:
print("Melhores parâmetros:", grid_search.best_params_)
print("Melhor score:", grid_search.best_score_)

Melhores parâmetros: {'model__batch_size': 32, 'model__epochs': 100, 'model__model__activation': 'tanh', 'model__model__dropout_rate': 0.3, 'model__model__neurons': 64, 'model__optimizer': 'rmsprop', 'pca__n_components': 150}
Melhor score: -0.15706683266225474


In [ ]:
y_pred_scaled = grid_search.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred)

np.float64(0.2520027740851982)

In [ ]:
best_model = grid_search.best_estimator_

new_movie = {
    'Series_Title': 'The Shawshank Redemption',
    'Released_Year': 1994,
    'Certificate': 'A',
    'Runtime': '142 min',
    'Genre': 'Drama',
    'Overview': 'Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.',
    'Meta_score': 80.0,
    'Director': 'Frank Darabont',
    'Star1': 'Tim Robbins',
    'Star2': 'Morgan Freeman',
    'Star3': 'Bob Gunton',
    'Star4': 'William Sadler',
    'No_of_Votes': 2343110,
    'Gross': '28,341,469'
}

df_new_movie = pd.DataFrame([new_movie])

df_new_movie.drop(labels=["Series_Title", "Overview"], axis=1, inplace=True)
df_new_movie['Gross'] = df_new_movie['Gross'].str.replace(',', '').astype(float)
df_new_movie["Runtime"] = df_new_movie["Runtime"].apply(modifies_runtime).astype(float)

try:
    df_new_movie_processed = preprocessor.transform(df_new_movie)
    y_new_pred_scaled = best_model.predict(df_new_movie_processed)
except NameError:
    print("AVISO: Preprocessor não encontrado. Verificando alternativas...")

    y_new_pred_scaled = best_model.predict(df_new_movie)

y_new_pred_original = scaler_y.inverse_transform(y_new_pred_scaled.reshape(-1, 1))

print(f"A previsão de IMDB Rating para o filme é: {y_new_pred_original[0][0]:.2f}")

A previsão de IMDB Rating para o filme é: 7.90
